In [1]:
#-------------------Task 1-------------------#

import heapq

def initial_board():
    return [
        ['w', 'w', 'w', 'w'],
        ['.', '.', '.', '.'],
        ['.', '.', '.', '.'],
        ['b', 'b', 'b', 'b']
    ]

def generate_moves(board, white_to_move):
    moves = []
    for r in range(4):
        for c in range(4):
            piece = board[r][c]
            if white_to_move and piece == 'w' and r < 3 and board[r+1][c] == '.':
                new_board = [row[:] for row in board]
                new_board[r][c] = '.'
                new_board[r+1][c] = 'w'
                moves.append(((r, c, r+1, c), new_board))
            elif not white_to_move and piece == 'b' and r > 0 and board[r-1][c] == '.':
                new_board = [row[:] for row in board]
                new_board[r][c] = '.'
                new_board[r-1][c] = 'b'
                moves.append(((r, c, r-1, c), new_board))
    return moves

def evaluate_board(board):
    white = sum(cell == 'w' for row in board for cell in row)
    black = sum(cell == 'b' for row in board for cell in row)
    return white - black

def beam_search_chess(board, beam_width=2, depth_limit=2, white_to_move=True):
    beam = [(0, [], board, white_to_move)]
    for depth in range(depth_limit):
        candidates = []
        for eval_score, move_seq, b, wtm in beam:
            moves = generate_moves(b, wtm)
            if not moves:
                candidates.append((eval_score, move_seq, b, wtm))
                continue
            for move, new_board in moves:
                score = evaluate_board(new_board)
                signed_score = score if wtm else -score
                candidates.append((-signed_score, move_seq + [move], new_board, not wtm))
        beam = heapq.nsmallest(beam_width, candidates, key=lambda x: x[0])
        if not beam:
            break
    if beam:
        best = min(beam, key=lambda x: x[0])
        return best[1], -best[0]
    return [], 0

def print_board(board):
    for row in board:
        print(' '.join(row))
    print()


board = initial_board()
beam_width = 3
depth_limit = 2
move_seq, eval_score = beam_search_chess(board, beam_width=beam_width, depth_limit=depth_limit, white_to_move=True)
print("Initial board:")
print_board(board)
if move_seq:
    print("Best move sequence:")
    b = [row[:] for row in board]
    wtm = True
    for move in move_seq:
        print(f"{'White' if wtm else 'Black'}: {move}")
        r1, c1, r2, c2 = move
        b[r2][c2] = b[r1][c1]
        b[r1][c1] = '.'
        print_board(b)
        wtm = not wtm
    print(f"Evaluation score: {eval_score}")
else:
    print("No move sequence found.")



Initial board:
w w w w
. . . .
. . . .
b b b b

Best move sequence:
White: (0, 0, 1, 0)
. w w w
w . . .
. . . .
b b b b

Black: (3, 0, 2, 0)
. w w w
w . . .
b . . .
. b b b

Evaluation score: 0


In [2]:
#-------------------Task 2-------------------#

import random
import math

def total_distance(route, points):
    dist = 0
    for i in range(len(route) - 1):
        x1, y1 = points[route[i]]
        x2, y2 = points[route[i+1]]
        dist += math.hypot(x2 - x1, y2 - y1)
    return dist

def hill_climbing_route(points):
    n = len(points)
    route = list(range(n))
    random.shuffle(route)
    best_distance = total_distance(route, points)
    improved = True

    while improved:
        improved = False
        for i in range(1, n-1):
            for j in range(i+1, n):
                new_route = route[:]
                new_route[i], new_route[j] = new_route[j], new_route[i]
                new_distance = total_distance(new_route, points)
                if new_distance < best_distance:
                    route = new_route
                    best_distance = new_distance
                    improved = True
    return route, best_distance

points = [
    (0, 0),
    (2, 3),
    (5, 4),
    (1, 1),
    (6, 2),
    (3, 5)
]
route, dist = hill_climbing_route(points)
print("Optimized route (by index):", route)
print("Route coordinates:", [points[i] for i in route])
print("Total distance:", dist)

Optimized route (by index): [4, 0, 3, 1, 5, 2]
Route coordinates: [(6, 2), (0, 0), (1, 1), (2, 3), (3, 5), (5, 4)]
Total distance: 14.446972815209223


In [3]:
#-------------------Task 3-------------------#

import random
import math

NUM_CITIES = 10

cities = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(NUM_CITIES)]

def distance(city1, city2):
	return math.hypot(city1[0] - city2[0], city1[1] - city2[1])

def total_distance(tour):
	return sum(distance(cities[tour[i]], cities[tour[(i+1)%NUM_CITIES]]) for i in range(NUM_CITIES))

POP_SIZE = 100
GENERATIONS = 500
MUTATION_RATE = 0.1
ELITE_SIZE = 5

def create_tour():
	tour = list(range(NUM_CITIES))
	random.shuffle(tour)
	return tour

def crossover(parent1, parent2):
	start, end = sorted(random.sample(range(NUM_CITIES), 2))
	child = [None]*NUM_CITIES
	child[start:end] = parent1[start:end]
	ptr = 0
	for city in parent2:
		if city not in child:
			while child[ptr] is not None:
				ptr += 1
			child[ptr] = city
	return child

def mutate(tour):
	if random.random() < MUTATION_RATE:
		i, j = random.sample(range(NUM_CITIES), 2)
		tour[i], tour[j] = tour[j], tour[i]
	return tour

def select(population, fitnesses):
	selected = []
	for _ in range(len(population)):
		i, j = random.sample(range(len(population)), 2)
		winner = population[i] if fitnesses[i] < fitnesses[j] else population[j]
		selected.append(winner)
	return selected

def genetic_algorithm():
	population = [create_tour() for _ in range(POP_SIZE)]
	for gen in range(GENERATIONS):
		fitnesses = [total_distance(tour) for tour in population]
		elite_indices = sorted(range(len(fitnesses)), key=lambda i: fitnesses[i])[:ELITE_SIZE]
		new_population = [population[i][:] for i in elite_indices]
		selected = select(population, fitnesses)
		while len(new_population) < POP_SIZE:
			p1, p2 = random.sample(selected, 2)
			child = crossover(p1, p2)
			child = mutate(child)
			new_population.append(child)
		population = new_population
		if (gen+1) % 100 == 0:
			print(f"Generation {gen+1}: Best distance = {min(fitnesses):.2f}")
	fitnesses = [total_distance(tour) for tour in population]
	best_idx = min(range(len(fitnesses)), key=lambda i: fitnesses[i])
	print("Best tour:", population[best_idx])
	print("Best distance:", fitnesses[best_idx])
	print("City coordinates:", cities)


genetic_algorithm()


Generation 100: Best distance = 288.45
Generation 200: Best distance = 286.47
Generation 300: Best distance = 286.47
Generation 400: Best distance = 286.47
Generation 500: Best distance = 286.47
Best tour: [9, 7, 1, 3, 6, 0, 8, 4, 2, 5]
Best distance: 286.4661030568963
City coordinates: [(11.299709585929662, 71.9265164244191), (22.03771659356071, 5.961757624003283), (48.14133362161839, 99.12298187763939), (2.4227764592859136, 60.2974015755881), (44.08950674342614, 91.62682823611034), (93.39467842346612, 61.208952532588214), (13.955232812548047, 60.560400495463675), (73.60054025438373, 48.159864180638614), (46.13451205906293, 81.96132324439176), (87.34874601295948, 52.21098899978529)]


In [4]:
#-------------------Task 4-------------------#

from typing import List, Tuple, Dict
import heapq

class Job:
	def __init__(self, job_id: int, exec_time: int, priority: int):
		self.job_id = job_id
		self.exec_time = exec_time
		self.priority = priority

class Processor:
	def __init__(self, proc_id: int):
		self.proc_id = proc_id
		self.load = 0
		self.jobs = []

def beam_search_allocate(jobs: List[Job], num_processors: int, beam_width: int = 3) -> Dict[int, List[int]]:
	initial_state = ([[] for _ in range(num_processors)], [0]*num_processors, set(range(len(jobs))))
	beam = [(score_state(initial_state, jobs), initial_state)]

	for _ in range(len(jobs)):
		next_beam = []
		for _, (assignments, loads, remaining) in beam:
			for job_idx in remaining:
				for proc_idx in range(num_processors):
					new_assignments = [a.copy() for a in assignments]
					new_loads = loads[:]
					new_remaining = set(remaining)
					new_assignments[proc_idx].append(job_idx)
					new_loads[proc_idx] += jobs[job_idx].exec_time
					new_remaining.remove(job_idx)
					new_state = (new_assignments, new_loads, new_remaining)
					heapq.heappush(next_beam, (score_state(new_state, jobs), new_state))
		beam = heapq.nsmallest(beam_width, next_beam)

	best_score, (best_assignments, _, _) = min(beam, key=lambda x: x[0])
	allocation = {i: [jobs[j].job_id for j in best_assignments[i]] for i in range(num_processors)}
	return allocation

def score_state(state, jobs: List[Job]) -> float:
	assignments, loads, remaining = state
	max_load = max(loads) if loads else 0
	priority_penalty = sum(jobs[j].priority for j in remaining)
	return max_load + 0.1 * priority_penalty


jobs = [Job(1, 5, 1), Job(2, 3, 2), Job(3, 2, 1), Job(4, 7, 3)]
num_processors = 2
allocation = beam_search_allocate(jobs, num_processors, beam_width=3)
print("Task allocation (processor: [job ids]):")
for proc, job_ids in allocation.items():
    print(f"Processor {proc}: {job_ids}")


Task allocation (processor: [job ids]):
Processor 0: [2, 4]
Processor 1: [3, 1]
